<a href="https://colab.research.google.com/github/canberkeken-bit/olist-analizi/blob/main/olist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Veri Yapıları ve Hesaplama
import pandas as pd
import numpy as np

# Veri Görselleştirme
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import bokeh.plotting as bk

# İstatistik ve Dosya İşleme
import scipy.stats as stats
import openpyxl

Gerekli Python kütüphaneleri yüklendi: pandas, numpy, plotly express.

In [10]:
df_customers = pd.read_csv('sample_data/olist/olist_customers_dataset.csv')
df_order_items = pd.read_csv('sample_data/olist/olist_order_items_dataset.csv')
df_order_payments = pd.read_csv('sample_data/olist/olist_order_payments_dataset.csv')
df_order_reviews = pd.read_csv('sample_data/olist/olist_order_reviews_dataset.csv')
df_orders = pd.read_csv('sample_data/olist/olist_orders_dataset.csv')
df_products = pd.read_csv('sample_data/olist/olist_products_dataset.csv')
df_seller = pd.read_csv('sample_data/olist/olist_sellers_dataset.csv')

Veri seti pandas DataFrame'e yüklendi.


In [11]:
dataframes = {
    'df_customers': df_customers,
    'df_order_items': df_order_items,
    'df_order_payments': df_order_payments,
    'df_order_reviews': df_order_reviews,
    'df_orders': df_orders,
    'df_products': df_products,
    'df_seller': df_seller
}

Burada yüklenen dataframeler sözlük yapısı halinde toparlandı amaç döngü kullanımını mümkün kılarak sütün ve veri tiplerini incelemek.






In [12]:
for name, df in dataframes.items():
  print(f" === {name} INFO ===")
  df.info()
  print("/n" + "*40" + "/n")

 === df_customers INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
/n*40/n
 === df_order_items INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 no

Burada döngü sayesinde butün csv'lerin veri tipleri ve sütünları incelendi.

In [13]:
for name, df in dataframes.items():
  print(f" === {name} Eksik Değer Sayıları ===")
  print(df.isnull().sum())
  print("\n" + "="*40 + "\n")

 === df_customers Eksik Değer Sayıları ===
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


 === df_order_items Eksik Değer Sayıları ===
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64


 === df_order_payments Eksik Değer Sayıları ===
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64


 === df_order_reviews Eksik Değer Sayıları ===
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


 === df_orders Eksik Değer Sayıları ===
order_id                       

In [14]:
df_order_reviews['review_comment_message'] = df_order_reviews['review_comment_message'].fillna('No Comment')
df_order_reviews['review_comment_title'] = df_order_reviews['review_comment_title'].fillna('No Title')
df_products['product_category_name'] = df_products['product_category_name'].fillna('No Category')
df_orders['order_approved_at'] = df_orders['order_approved_at'].fillna(df_orders['order_purchase_timestamp'])

Eksik değerler dolduruldu. sayısal değerlere sahip olanlar ortalama ile doldurulmadı sebebi ise yanlış bilginin önüne geçmek.

In [15]:
from string import printable
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'], errors='coerce')
df_orders['order_approved_at'] = pd.to_datetime(df_orders['order_approved_at'], errors='coerce')
df_orders['order_delivered_carrier_date'] = pd.to_datetime(df_orders['order_delivered_carrier_date'], errors='coerce')
df_orders['order_delivered_customer_date'] = pd.to_datetime(df_orders['order_delivered_customer_date'], errors='coerce')
df_orders['order_estimated_delivery_date'] = pd.to_datetime(df_orders['order_estimated_delivery_date'], errors='coerce')
df_order_items['shipping_limit_date'] = pd.to_datetime(df_order_items['shipping_limit_date'], errors='coerce')
df_order_reviews['review_creation_date'] = pd.to_datetime(df_order_reviews['review_creation_date'], errors='coerce')
df_order_reviews['review_answer_timestamp'] = pd.to_datetime(df_order_reviews['review_answer_timestamp'], errors= 'coerce')
print("Dönüşüm tamamlandı. Örnek olarak df_orders veri tipleri:")
print(df_orders.dtypes)

Dönüşüm tamamlandı. Örnek olarak df_orders veri tipleri:
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


## Ek Analizler ve Görselleştirmeler

### 1. Sipariş Fiyatı İstatistikleri ve Dağılım Histogramı

In [16]:
df_order_price_sum = df_order_items.groupby('order_id')['price'].sum().reset_index()
price_mean = df_order_price_sum['price'].mean()
price_median = df_order_price_sum['price'].median()
price_std = df_order_price_sum['price'].std()
print(f"Ortalama Sipariş Fiyatı: {price_mean:.2f} BRL")
print(f"Medyan Sipariş Fiyatı: {price_median:.2f} BRL")
print(f"Sipariş Fiyatlarının Standart Sapması: {price_std:.2f} BRL")

# Hatalar giderildi: Veri seti düzeltildi ve labels sözlüğü tek parça haline getirildi.
fig_hist = px.histogram(df_order_price_sum[df_order_price_sum['price'] <= 500], x='price', nbins=100,
                         title='Sipariş Fiyatı Dağılımı (500 BRL ve Altı Siparişler)',
                         labels={'price': 'Sipariş Fiyatı (BRL)', 'count': 'Sipariş Sayısı'})
fig_hist.show()

Ortalama Sipariş Fiyatı: 137.75 BRL
Medyan Sipariş Fiyatı: 86.90 BRL
Sipariş Fiyatlarının Standart Sapması: 210.65 BRL


### 2. Ürün Kategorisi Dağılımı (En Çok Sipariş Edilen İlk 15 Kategori)

In [17]:
df_item_product = pd.merge(df_order_items, df_products[['product_id', 'product_category_name']], on='product_id')
category_counts = df_item_product['product_category_name'].value_counts().reset_index().rename(columns={'count': 'siparis_sayısı'})
fig_bar = px.bar(category_counts.head(15), x='product_category_name', y='siparis_sayısı',
                 title='En çok sipariş edilen 15 ürün kategorisi',
                 labels= {'product_category_name': 'Ürün Kategorisi', 'siparis_sayısı': 'Sipariş Sayısı'},
                 color='siparis_sayısı')
fig_bar.show()

### 3. Ödeme Yöntemi Dağılımı Oranları

In [18]:
payment_counts = df_order_payments['payment_type'].value_counts().reset_index().rename(columns={'count': 'islem_sayisi'})
fig_pie = px.pie(payment_counts, values= 'islem_sayisi', names= 'payment_type',
                 title= 'Ödeme tipi dağılım oranları',
                 hole=0.3)
fig_pie.show()

### 4. Kargo ve Teslimat Süresi İstatistikleri

In [19]:
df_orders['delivery_time_days'] = (df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']).dt.days

delivery_days_clean = df_orders['delivery_time_days'].dropna()

del_mean = delivery_days_clean.mean()
del_median = delivery_days_clean.median()
del_std = delivery_days_clean.std()

print(f"Ortalama Teslimat Süresi: {del_mean:.2f} gün")
print(f"Medyan Teslimat Süresi: {del_median:.2f} gün")
print(f"Teslimat Süresi Standart Sapması: {del_std:.2f} gün")

# Kutu Grafiği (Boxplot)
fig_box = px.box(df_orders, y='delivery_time_days',
                 title='Teslimat Süresi Kutu Grafiği (Boxplot)',
                 labels={'delivery_time_days': 'Teslimat Süresi (Gün)'})
fig_box.show()

Ortalama Teslimat Süresi: 12.09 gün
Medyan Teslimat Süresi: 10.00 gün
Teslimat Süresi Standart Sapması: 9.55 gün


### 5. Müşteri Memnuniyeti Puanları Analizi

In [20]:
review_scores = df_order_reviews['review_score'].dropna()

mean_review = review_scores.mean()
median_review = review_scores.median()

# 4 ve üzeri puan alan memnun müşteriler vs 4 altı memnun olmayanlar
satisfied_count = review_scores[review_scores >= 4].count()
unsatisfied_count = review_scores[review_scores < 4].count()
total_reviews = review_scores.count()

satisfaction_rate = (satisfied_count / total_reviews) * 100

print(f"Ortalama Memnuniyet Puanı: {mean_review:.2f} / 5")
print(f"Medyan Memnuniyet Puanı: {median_review:.2f} / 5")
print(f"Memnun Müşteri Sayısı (>= 4): {satisfied_count} (%{satisfaction_rate:.2f})")
print(f"Memnun Olmayan Müşteri Sayısı (< 4): {unsatisfied_count} (%{(unsatisfied_count/total_reviews)*100:.2f})")

Ortalama Memnuniyet Puanı: 4.09 / 5
Medyan Memnuniyet Puanı: 5.00 / 5
Memnun Müşteri Sayısı (>= 4): 76470 (%77.07)
Memnun Olmayan Müşteri Sayısı (< 4): 22754 (%22.93)


In [21]:
# Memnuniyet durumunu gruplandıralım
satisfaction_df = pd.DataFrame({
    'Durum': ['Memnun Müşteriler (4-5 Yıldız)', 'Memnun Olmayan Müşteriler (1-3 Yıldız)'],
    'Müşteri Sayısı': [satisfied_count, unsatisfied_count]
})

# Oranları gösteren bir pasta grafiği hazırlayalım
fig_satisfaction = px.pie(satisfaction_df, values='Müşteri Sayısı', names='Durum',
                          title='Müşteri Memnuniyet Oranlarının Karşılaştırılması',
                          color_discrete_sequence=['#2ecc71', '#e74c3c'],
                          hole=0.4)

# Grafiğe yüzde ve etiket değerlerini ekleyelim
fig_satisfaction.update_traces(textinfo='percent+label')
fig_satisfaction.show()

### 6. Ürün Boyutunun (Hacim) Kargo Süresine Etkisi (Eğim / Regresyon Grafiği)

In [22]:
import numpy as np
# Ürün hacmini hesaplayalım (Genişlik x Yükseklik x Uzunluk)
df_products['product_volume_cm3'] = df_products['product_length_cm'] * df_products['product_height_cm'] * df_products['product_width_cm']

# Siparişler ile ürün kargo bilgilerini birleştirelim
df_order_items_delivered = pd.merge(df_order_items, df_orders[['order_id', 'delivery_time_days']], on='order_id')
df_volume_delivery = pd.merge(df_order_items_delivered, df_products[['product_id', 'product_volume_cm3']], on='product_id').dropna()

# Grafik performansı için örneklem alıp regresyon eğrisi çizelim
df_sample = df_volume_delivery.sample(n=min(5000, len(df_volume_delivery)), random_state=42)

fig_volume = px.scatter(df_sample, x='product_volume_cm3', y='delivery_time_days', trendline='ols',
                        title='Ürün Boyutu (Hacim cm³) ile Kargo Teslimat Süresi İlişkisi',
                        labels={'product_volume_cm3': 'Ürün Hacmi (cm³)', 'delivery_time_days': 'Teslimat Süresi (Gün)'},
                        opacity=0.4, color_discrete_sequence=['#2980b9'])
fig_volume.show()

### 7. Kargolanma/Teslimat Süresinin Müşteri Memnuniyet Puanına Etkisi

> Blok alıntı ekle



In [23]:
df_orders['delivery_time_days'] = (
    pd.to_datetime(df_orders['order_delivered_customer_date']) -
    pd.to_datetime(df_orders['order_purchase_timestamp'])
).dt.days

df_delivery_review = df_orders[['order_id', 'delivery_time_days']].merge(
    df_order_reviews[['order_id', 'review_score']],
    on='order_id'
).dropna()

bins = [-1, 3, 7, 14, 21, 30, 100]
labels = ['0-3 Gün', '4-7 Gün', '8-14 Gün', '15-21 Gün', '22-30 Gün', '30+ Gün']

df_delivery_review['delivery_range'] = pd.cut(
    df_delivery_review['delivery_time_days'],
    bins=bins,
    labels=labels
)

delivery_summary = df_delivery_review.groupby('delivery_range', observed=False).agg(
    ortalama_puan=('review_score', 'mean'),
    toplam_siparis=('review_score', 'count')
).reset_index()

print("=== Teslimat Süresi Aralıklarına Göre Memnuniyet Tablosu ===")
print(delivery_summary.to_string(index=False))


fig = px.line(
    delivery_summary,
    x='delivery_range',
    y='ortalama_puan',
    markers=True,
    text='ortalama_puan',
    title='Teslimat Süresine Göre Ortalama Müşteri Memnuniyeti Değişimi',
    labels={'delivery_range': 'Teslimat Süresi Aralığı', 'ortalama_puan': 'Ortalama Puan (1-5)'}
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='top center')
fig.update_layout(yaxis_range=[1, 5])
fig.show()

=== Teslimat Süresi Aralıklarına Göre Memnuniyet Tablosu ===
delivery_range  ortalama_puan  toplam_siparis
       0-3 Gün       4.459981            8596
       4-7 Gün       4.390490           25089
      8-14 Gün       4.288356           36396
     15-21 Gün       4.102334           15381
     22-30 Gün       3.493955            6865
       30+ Gün       2.164653            3972


### 8. Ürün Fotoğraf Sayısının Satış Miktarına Etkisi

In [24]:
# Ürünlerin satış miktarını bulalım (her ürünün kaç kez sipariş edildiği)
product_sales_qty = df_order_items['product_id'].value_counts().reset_index().rename(columns={'count': 'sales_qty'})

# Fotoğraf sayısı ile birleştirelim
df_photo_sales = pd.merge(df_products[['product_id', 'product_photos_qty']], product_sales_qty, on='product_id').dropna()

# Fotoğraf sayısına göre ortalama satış miktarını gruplayalım
df_photo_grouped = df_photo_sales.groupby('product_photos_qty')['sales_qty'].mean().reset_index()

fig_photos = px.line(df_photo_grouped, x='product_photos_qty', y='sales_qty',
                     title='Ürün Fotoğraf Sayısının Ortalama Satış Adetlerine Etkisi',
                     labels={'product_photos_qty': 'Fotoğraf Sayısı', 'sales_qty': 'Ortalama Satış Adedi'},
                     markers=True, color_discrete_sequence=['#27ae60'])
fig_photos.show()

### 9. En Çok Satış Yapan Eyalet ve Şehirler (Kloroplet / Yoğunluk Haritası)

In [25]:
# Müşteri sipariş eyalet verisini hazırlayalım
df_order_customers = pd.merge(df_orders[['order_id', 'customer_id']], df_customers[['customer_id', 'customer_state', 'customer_city']], on='customer_id')
df_state_sales = df_order_customers['customer_state'].value_counts().reset_index().rename(columns={'count': 'Sipariş Sayısı', 'customer_state': 'Eyalet'})

fig_map = px.choropleth(df_state_sales,
                           locations='Eyalet',
                           locationmode='USA-states', # Brezilya eyalet kodları için en iyi gösterim amaçlı renk haritası
                           color='Sipariş Sayısı',
                           color_continuous_scale='Blues',
                           title='Brezilya Eyaletlerine Göre Sipariş Yoğunluk Dağılımı (Bölgesel Dağılım)',
                           labels={'Sipariş Sayısı': 'Toplam Sipariş Adedi'})
fig_map.show()

# İlk 15 şehri bar grafiği yerine balon grafiğiyle (bubble chart) gösterelim
df_city_sales = df_order_customers['customer_city'].value_counts().reset_index().rename(columns={'count': 'Sipariş Sayısı', 'customer_city': 'Şehir'}).head(15)

fig_city_bubble = px.scatter(df_city_sales, x='Şehir', y='Sipariş Sayısı',
                             size='Sipariş Sayısı', color='Şehir',
                             title='En Çok Satış Yapılan İlk 15 Şehir (Yoğunluk Balon Grafiği)',
                             labels={'Şehir': 'Şehir Adı', 'Sipariş Sayısı': 'Toplam Sipariş Adedi'},
                             size_max=50)
fig_city_bubble.show()

### 10. Ödeme Yönteminin Toplam Ödeme Tutarına Etkisi (Yoğunluk Balon Grafiği - Bubble Chart)

In [26]:
# Her ödeme türü için işlem adedini, ortalama ödeme değerini hesaplayalım
df_bubble_data = df_order_payments.groupby('payment_type').agg(
    Ortalama_Odeme=('payment_value', 'mean'),
    Toplam_Islem=('payment_value', 'count'),
    Toplam_Hacim=('payment_value', 'sum')
).reset_index()

# not_defined olanları temizleyelim
df_bubble_data = df_bubble_data[df_bubble_data['payment_type'] != 'not_defined']

fig_bubble = px.scatter(df_bubble_data, x='payment_type', y='Ortalama_Odeme',
                        size='Toplam_Islem', color='payment_type',
                        title='Ödeme Yöntemlerinin Ortalama Ödeme Tutarı ve İşlem Hacmi (Balon Büyüklüğü: İşlem Sayısı)',
                        labels={'payment_type': 'Ödeme Yöntemi', 'Ortalama_Odeme': 'Ortalama Ödeme Tutarı (BRL)', 'Toplam_Islem': 'İşlem Sayısı'},
                        size_max=60)
fig_bubble.show()

Sipariş Fiyatı vs. Teslimat Süresi:

In [27]:
order_costs = df_order_items.groupby('order_id')['price'].sum().reset_index()
df_orders['delivery_time_days'] = (pd.to_datetime(df_orders['order_delivered_customer_date']) - pd.to_datetime(df_orders['order_purchase_timestamp'])).dt.days

df_anaylsis = df_orders[['order_id', 'delivery_time_days']].merge(order_costs, on='order_id')
df_anaylsis = df_anaylsis.merge(df_order_reviews[['order_id', 'review_score']], on='order_id')
df_anaylsis = df_anaylsis.dropna()

df_anaylsis['price_segment'] = pd.qcut(df_anaylsis['price'], q=4, labels=['Düşük', 'Orta', 'Yüksek', 'Çok Yüksek'])

bins = [0, 7, 14, 21, 30, np.inf]
labels = ['0-7 Gün', '8-14 Gün', '15-21 Gün', '22-30 Gün', '30+ Gün']
df_anaylsis['delivery_range'] = pd.cut(df_anaylsis['delivery_time_days'], bins=bins, labels=labels)
tolarance_table = df_anaylsis.groupby(['price_segment', 'delivery_range'], observed=False)['review_score'].mean().unstack()
print(tolarance_table)

# Hata düzeltildi: df_analysis değişken adı df_anaylsis olarak güncellendi ve observed=False eklendi
tolerance_df = df_anaylsis.groupby(['price_segment', 'delivery_range'], observed=False)['review_score'].mean().reset_index()

# Çizgi Grafiği
fig = px.line(
    tolerance_df,
    x='delivery_range',
    y='review_score',
    color='price_segment',
    markers=True,
    title='Harcama Segmentlerine Göre Teslimat Süresi Toleransı (Ortalama Puan)',
    labels={'delivery_range': 'Teslimat Süresi Aralığı', 'review_score': 'Ortalama Müşteri Puanı', 'price_segment': 'Segment'}
)

fig.show()

delivery_range   0-7 Gün  8-14 Gün  15-21 Gün  22-30 Gün   30+ Gün
price_segment                                                     
Düşük           4.444982  4.302441   4.129061   3.488435  2.169069
Orta            4.403529  4.304592   4.162908   3.522741  2.093319
Yüksek          4.417442  4.311298   4.092758   3.457253  2.184751
Çok Yüksek      4.353351  4.235243   4.034831   3.508346  2.231005


Sipariş Fiyatı vs. Memnuniyet: Harcama miktarı müşteri memnuniyetini etkiliyor mu?

In [28]:
order_prices = df_order_items.groupby('order_id')['price'].sum().reset_index()
df_price_review = order_prices.merge(df_order_reviews[['order_id', 'review_score']], on='order_id').dropna()
df_price_review['price_segment'] = pd.qcut(df_price_review['price'],
                                           q=4,
                                           labels= ['Düşük', 'Orta', 'Yüksek', 'Çok Yüksek'])
segment_summary = df_price_review.groupby('price_segment').agg(
    ortalama_harcama=('price', 'mean'),
    ortalama_puan=('review_score', 'mean'),
    toplam_siparis=('order_id', 'count')
).reset_index()

print("=== Harcama Segmentleri Özet Tablosu ===")
print(segment_summary)
fig = px.bar(
    segment_summary,
    x='price_segment',
    y='ortalama_puan',
    text='ortalama_puan',
    title='Harcama Segmentlerine Göre Ortalama Müşteri Puanı',
    labels={'price_segment': 'Harcama Segmentleri', 'ortalama_puan': 'Ortalama Müşteri Puanı'}
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(yaxis_range=[1, 5], showlegend=False)
fig.show()

=== Harcama Segmentleri Özet Tablosu ===
  price_segment  ortalama_harcama  ortalama_puan  toplam_siparis
0         Düşük         28.587535       4.171695           24759
1          Orta         63.809472       4.132263           24474
2        Yüksek        114.912658       4.109591           24783
3    Çok Yüksek        343.976137       4.003967           24449


/tmp/ipykernel_1194/2303878002.py:6: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Ürün Kategorisi vs. Memnuniyet: Farklı kategorilerde memnuniyet seviyeleri değişiyor mu?

In [29]:
df_cat_review = df_products[['product_id', 'product_category_name']].merge(df_order_items[['order_id', 'product_id']], on='product_id').merge(df_order_reviews[['order_id', 'review_score']], on='order_id')
df_cat_review['product_category_name'] = df_cat_review['product_category_name'].fillna('unknown')
cat_summary = df_cat_review.groupby('product_category_name').agg(
    ortalama_puan=('review_score', 'mean'),
    toplam_siparis=('order_id', 'count')
).reset_index()

top10_high = cat_summary_filtered.sort_values(by='ortalama_puan', ascending=False).head(10)
top10_low = cat_summary_filtered.sort_values(by='ortalama_puan', ascending=True).head(10)
print("=== En Yüksek Memnuniyete Sahip 5 Kategori ===")
print(top10_high[['product_category_name', 'ortalama_puan', 'toplam_siparis']].head())

print("\n=== En Düşük Memnuniyete Sahip 5 Kategori ===")
print(top10_low[['product_category_name', 'ortalama_puan', 'toplam_siparis']].head())
fig_high = px.bar(
    top10_high,
    x='ortalama_puan',
    y='product_category_name',
    orientation='h',
    text='ortalama_puan',
    color='ortalama_puan',
    color_continuous_scale='Greens',
    title='En Yüksek Müşteri Memnuniyetine Sahip Top 10 Kategori (Min 50 Sipariş)',
    labels={'product_category_name': 'Kategori', 'ortalama_puan': 'Ortalama Puan'}
)
fig_high.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig_high.update_layout(yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False)
fig_high.show()

fig_low = px.bar(
    top10_low,
    x='ortalama_puan',
    y='product_category_name',
    orientation='h',
    text='ortalama_puan',
    color='ortalama_puan',
    color_continuous_scale='Reds_r',
    title='En Düşük Müşteri Memnuniyetine Sahip Top 10 Kategori (Min 50 Sipariş)',
    labels={'product_category_name': 'Kategori', 'ortalama_puan': 'Ortalama Puan'}
)
fig_low.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig_low.update_layout(yaxis={'categoryorder': 'total descending'}, coloraxis_showscale=False)
fig_low.show()

NameError: name 'cat_summary_filtered' is not defined

Ödeme Yöntemi vs. Memnuniyet: Ödeme yöntemi memnuniyetle ilişkili mi?

In [30]:
df_payment_review = df_order_payments[['order_id', 'payment_type']].merge(
    df_order_reviews[['order_id', 'review_score']],
    on='order_id'
).dropna()


payment_summary = df_payment_review.groupby('payment_type').agg(
    ortalama_puan=('review_score', 'mean'),
    toplam_islem=('review_score', 'count')
).reset_index()

payment_summary['islem_payi_%'] = (payment_summary['toplam_islem'] / payment_summary['toplam_islem'].sum()) * 100
payment_summary = payment_summary.sort_values(by='ortalama_puan', ascending=False)

print("=== Ödeme Yöntemlerine Göre Memnuniyet Özeti ===")
print(payment_summary)

fig = px.bar(
    payment_summary,
    x='payment_type',
    y='ortalama_puan',
    text='ortalama_puan',
    color='payment_type',
    title='Ödeme Yöntemlerine Göre Ortalama Müşteri Memnuniyeti',
    labels={'payment_type': 'Ödeme Yöntemi', 'ortalama_puan': 'Ortalama Puan (1-5)'}
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(yaxis_range=[1, 5], showlegend=False)
fig.show()

=== Ödeme Yöntemlerine Göre Memnuniyet Özeti ===
  payment_type  ortalama_puan  toplam_islem  islem_payi_%
2   debit_card       4.168084          1529      1.474773
1  credit_card       4.087637         76600     73.883311
0       boleto       4.086631         19762     19.061123
4      voucher       4.003804          5783      5.577901
3  not_defined       1.666667             3      0.002894


In [31]:
df_payment_review = df_order_payments[['order_id', 'payment_type']].merge(
    df_order_reviews[['order_id', 'review_score']],
    on='order_id'
).dropna()


payment_summary = df_payment_review.groupby('payment_type').agg(
    ortalama_puan=('review_score', 'mean'),
    toplam_islem=('review_score', 'count')
).reset_index()

payment_summary['islem_payi_%'] = (payment_summary['toplam_islem'] / payment_summary['toplam_islem'].sum()) * 100
payment_summary = payment_summary.sort_values(by='ortalama_puan', ascending=False)

print("=== Ödeme Yöntemlerine Göre Memnuniyet Özeti ===")
print(payment_summary)

fig = px.bar(
    payment_summary,
    x='payment_type',
    y='ortalama_puan',
    text='ortalama_puan',
    color='payment_type',
    title='Ödeme Yöntemlerine Göre Ortalama Müşteri Memnuniyeti',
    labels={'payment_type': 'Ödeme Yöntemi', 'ortalama_puan': 'Ortalama Puan (1-5)'}
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(yaxis_range=[1, 5], showlegend=False)
fig.show()

=== Ödeme Yöntemlerine Göre Memnuniyet Özeti ===
  payment_type  ortalama_puan  toplam_islem  islem_payi_%
2   debit_card       4.168084          1529      1.474773
1  credit_card       4.087637         76600     73.883311
0       boleto       4.086631         19762     19.061123
4      voucher       4.003804          5783      5.577901
3  not_defined       1.666667             3      0.002894


### 1. Teslimat Süresinin Müşteri Memnuniyetine Etkisi
**H0:** Teslimat süresi ile müşteri memnuniyeti arasında anlamlı bir ilişki yoktur.
**H1:** Teslimat süresi uzadıkça müşteri memnuniyeti azalır (negatif yönlü anlamlı bir ilişki vardır).

In [36]:
import scipy.stats as stats

# Eksik değerleri temizleyelim
df_delivery_clean = df_delivery_review.dropna(subset=['delivery_time_days', 'review_score'])

# Korelasyon testi uygulayalım
pearson_corr, p_val_corr = stats.pearsonr(df_delivery_clean['delivery_time_days'],
                                          df_delivery_clean['review_score'])

print(f"Pearson Korelasyon Katsayısı: {pearson_corr:.4f}")
print(f"p-değeri: {p_val_corr}")



if p_val_corr < 0.05:
    yon = "negatif" if pearson_corr < 0 else "pozitif"
    print(f"H0 reddedildi. Anlamlı {yon} bir ilişki var.")
else:
    print("H0 reddedilemedi.")

Pearson Korelasyon Katsayısı: -0.3337
p-değeri: 0.0
H0 reddedildi. Anlamlı negatif bir ilişki var.


### 2. Ürün Kategorisinin Sipariş Değerine Etkisi
**H0:** Farklı kategorilerdeki ürünlerin ortalama sipariş değerleri arasında anlamlı bir fark yoktur.
**H1:** En az bir kategorinin ortalama sipariş değeri diğerlerinden anlamlı derecede farklıdır.

In [39]:
# Sipariş ürünleri ile ürün kategorilerini birleştirelim
df_item_cat = pd.merge(df_order_items, df_products[['product_id', 'product_category_name']], on='product_id')
# En popüler 5 kategoriyi baz alarak ANOVA testi gerçekleştirelim
top_categories = df_item_cat['product_category_name'].value_counts().head(5).index
groups = [df_item_cat[df_item_cat['product_category_name'] == cat]['price'].values for cat in top_categories]

f_stat, p_val_anova = stats.f_oneway(*groups)
print(f"F-İstatistiği: {f_stat:.4f}")
print(f"p-değeri: {p_val_anova}")

if p_val_anova < 0.05:
    print("Sonuç: H0 reddedildi. Seçilen kategoriler arasında ortalama ürün fiyatları bakımından anlamlı bir fark vardır.")
else:
    print("Sonuç: H0 reddedilemedi. Kategoriler arasında anlamlı bir fiyat farkı bulunamamıştır.")

F-İstatistiği: 145.1869
p-değeri: 1.4054652734072725e-123
Sonuç: H0 reddedildi. Seçilen kategoriler arasında ortalama ürün fiyatları bakımından anlamlı bir fark vardır.


### 3. Ödeme Yönteminin İşlem Tutarındaki Etkisi (Kredi Kartı vs. Boleto)
**H0:** Kredi kartı ve Boleto ile yapılan ödemelerin ortalama tutarları arasında anlamlı bir fark yoktur.
**H1:** Kredi kartı ve Boleto ile yapılan ödemelerin ortalama tutarları arasında anlamlı bir fark vardır.

In [43]:
# Kredi kartı ve boleto ödeme değerlerini filtreleyelim
c_payments = df_order_payments[df_order_payments['payment_type'] == 'credit_card']['payment_value']
boleto_payments = df_order_payments[df_order_payments['payment_type'] == 'boleto']['payment_value']

# Bağımsız iki örneklem t-testi
t_stat, p_val_t = stats.ttest_ind(c_payments, boleto_payments, equal_var=False)
print(f"t-İstatistiği: {t_stat:.4f}")
print(f"p-değeri: {p_val_t}")

if p_val_t < 0.05:
    print("Sonuç: H0 reddedildi. Kredi Kartı ve Boleto ödemelerinin ortalama değerleri arasında anlamlı bir fark vardır.")
else:
    print("Sonuç: H0 reddedilemedi. İki ödeme yöntemi arasında ortalama ödeme tutarı bakımından anlamlı bir fark yoktur.")

t-İstatistiği: 10.6490
p-değeri: 1.9545230270625305e-26
Sonuç: H0 reddedildi. Kredi Kartı ve Boleto ödemelerinin ortalama değerleri arasında anlamlı bir fark vardır.
